In [9]:
# Final cleaned and analysis-ready dataset
print("="*60)
print("FINAL ANALYSIS-READY DATASET SUMMARY")
print("="*60)

print(f"\nShape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
print(f"\nData Types:\n{df_clean.dtypes}")
print(f"\nMissing Values:\n{df_clean.isnull().sum()}")

print("\n" + "="*60)
print("KEY STATISTICS:")
print("="*60)
print(f"Total Customers: {len(df_clean)}")
print(f"Countries Represented: {df_clean['Country'].nunique()}")
print(f"Date Range: {df_clean['Subscription Date'].min().date()} to {df_clean['Subscription Date'].max().date()}")
print(f"Years Covered: {sorted(df_clean['Subscription Year'].unique())}")

print("\n" + "="*60)
print("Sample of cleaned data (first 5 rows):")
print("="*60)
display_cols = ['Customer Id', 'First Name', 'Last Name', 'Country', 'Subscription Date', 'Subscription Year']
print(df_clean[display_cols].head())

FINAL ANALYSIS-READY DATASET SUMMARY

Shape: (95, 14)
Columns: ['Index', 'Customer Id', 'First Name', 'Last Name', 'Company', 'City', 'Country', 'Phone 1', 'Phone 2', 'Email', 'Subscription Date', 'Website', 'Subscription Year', 'Subscription Month']

Data Types:
Index                          int64
Customer Id                      str
First Name                       str
Last Name                        str
Company                          str
City                             str
Country                          str
Phone 1                          str
Phone 2                          str
Email                            str
Subscription Date     datetime64[us]
Website                          str
Subscription Year              int32
Subscription Month             int32
dtype: object

Missing Values:
Index                 0
Customer Id           0
First Name            0
Last Name             0
Company               0
City                  0
Country               0
Phone 1            

## 8. Final Summary - Analysis-Ready Dataset

In [8]:
# Pivot Table 1: Subscriptions by Year and Month
pivot_year_month = pd.pivot_table(
    df_clean,
    values='Customer Id',
    index='Subscription Year',
    columns='Subscription Month',
    aggfunc='count',
    fill_value=0
)
print("Subscriptions by Year and Month (Pivot Table):")
print(pivot_year_month)

print("\n" + "="*60 + "\n")

# Pivot Table 2: Count of customers by Country and Year
pivot_country_year = pd.pivot_table(
    df_clean,
    values='Customer Id',
    index='Country',
    columns='Subscription Year',
    aggfunc='count',
    fill_value=0
)
print("Customers by Country and Subscription Year:")
print(pivot_country_year.sort_values(by=2021, ascending=False).head(10))

print("\n" + "="*60 + "\n")

# Pivot Table 3: With margins (totals)
pivot_with_margins = pd.pivot_table(
    df_merged[df_merged['Region'].notna()],  # Use merged data with Region
    values='Customer Id',
    index='Region',
    columns='Subscription Year',
    aggfunc='count',
    margins=True,
    fill_value=0
)
print("Customers by Region and Year (with totals):")
print(pivot_with_margins)

Subscriptions by Year and Month (Pivot Table):
Subscription Month  1   2   3   4   5   6   7   8   9   10  11  12
Subscription Year                                                 
2020                 2   5   3   2   1   2   4   3   4   3   3   4
2021                 4   3   5   8   0   1   3   3   6   1   3   3
2022                 7   2   4   1   5   0   0   0   0   0   0   0


Customers by Country and Subscription Year:
Subscription Year  2020  2021  2022
Country                            
Solomon Islands       0     3     1
Sri Lanka             0     2     0
Albania               0     1     0
Algeria               0     1     0
Belarus               0     1     1
Bahamas               0     1     0
Anguilla              0     1     0
Aruba                 0     1     0
Bulgaria              0     1     1
Hungary               0     1     0


Customers by Region and Year (with totals):
Subscription Year  2020  2021  2022  All
Region                                  
Asia        

## 7. Pivot Tables

In [7]:
# Create a sample "Country Regions" dataframe to demonstrate merging
country_regions = pd.DataFrame({
    'Country': ['Chile', 'Vietnam', 'Bosnia and Herzegovina', 'Bulgaria', 'Cyprus'],
    'Region': ['South America', 'Asia', 'Europe', 'Europe', 'Europe'],
    'Code': ['CL', 'VN', 'BA', 'BG', 'CY']
})

print("Country Regions Dataset:")
print(country_regions)

print("\n" + "="*60 + "\n")

# Merge customers with country regions
df_merged = pd.merge(df_clean, country_regions, on='Country', how='left')

print(f"Merged dataset shape: {df_merged.shape}")
print("\nMerged data sample:")
print(df_merged[['First Name', 'Country', 'Region']].head(10))

print("\n" + "="*60 + "\n")

# Check for customers from regions with data
print("Customers with Region information:")
print(df_merged[df_merged['Region'].notna()][['First Name', 'Country', 'Region']].head())

Country Regions Dataset:
                  Country         Region Code
0                   Chile  South America   CL
1                 Vietnam           Asia   VN
2  Bosnia and Herzegovina         Europe   BA
3                Bulgaria         Europe   BG
4                  Cyprus         Europe   CY


Merged dataset shape: (95, 16)

Merged data sample:
  First Name                     Country  Region
0    Preston                    Djibouti     NaN
1        Roy         Antigua and Barbuda     NaN
2      Linda          Dominican Republic     NaN
3     Joanna  Slovakia (Slovak Republic)     NaN
4      Aimee      Bosnia and Herzegovina  Europe
5     Darren            Pitcairn Islands     NaN
6      Brett                    Bulgaria  Europe
7     Sheryl                      Cyprus  Europe
8   Michelle                 Timor-Leste     NaN
9      Jenna                     Vietnam    Asia


Customers with Region information:
   First Name                 Country  Region
4       Aimee  Bosnia a

## 6. Merging Datasets

In [6]:
# GroupBy 1: Customers by Country
customers_by_country = df_clean.groupby('Country').size().sort_values(ascending=False)
print("Customers by Country (Top 10):")
print(customers_by_country.head(10))

print("\n" + "="*60 + "\n")

# GroupBy 2: Multiple aggregations
country_stats = df_clean.groupby('Country').agg({
    'Customer Id': 'count',  # Total customers
    'Subscription Year': ['min', 'max'],  # Year range
    'City': 'nunique'  # Number of unique cities
}).round(2)
country_stats.columns = ['Total Customers', 'First Subscription', 'Latest Subscription', 'Unique Cities']
print("Country Statistics (Top 10):")
print(country_stats.sort_values('Total Customers', ascending=False).head(10))

print("\n" + "="*60 + "\n")

# GroupBy 3: Subscriptions by Year
subscriptions_by_year = df_clean.groupby('Subscription Year').agg({
    'Customer Id': 'count',
    'First Name': 'count'
}).rename(columns={'Customer Id': 'Number of Subscriptions', 'First Name': 'Customers'})
print("Subscriptions by Year:")
print(subscriptions_by_year)

Customers by Country (Top 10):
Country
Solomon Islands         4
Belarus                 2
Bulgaria                2
Canada                  2
Sri Lanka               2
Togo                    2
Zimbabwe                2
Oman                    2
United Arab Emirates    2
Netherlands             2
dtype: int64


Country Statistics (Top 10):
                      Total Customers  First Subscription  \
Country                                                     
Solomon Islands                     4                2021   
Belarus                             2                2021   
Bulgaria                            2                2021   
Canada                              2                2020   
Sri Lanka                           2                2021   
Togo                                2                2020   
Zimbabwe                            2                2020   
Oman                                2                2021   
United Arab Emirates                2          

## 5. GroupBy Operations

In [5]:
# Filter 1: Customers from specific years
recent_customers = df_clean[df_clean['Subscription Year'] >= 2021]
print(f"Customers subscribed in 2021 or later: {len(recent_customers)}")
print(recent_customers[['First Name', 'Country', 'Subscription Date']].head())

print("\n" + "="*60 + "\n")

# Filter 2: Customers from specific countries
target_countries = ['Chile', 'Vietnam', 'Bosnia and Herzegovina']
customers_target_countries = df_clean[df_clean['Country'].isin(target_countries)]
print(f"Customers from {target_countries}:")
print(customers_target_countries[['First Name', 'Country']].head())

print("\n" + "="*60 + "\n")

# Filter 3: Customers with Email provided and from 2020
filtered_df = df_clean[(df_clean['Email'] != '') & (df_clean['Subscription Year'] == 2020)]
print(f"Customers from 2020 with email: {len(filtered_df)}")
print(filtered_df[['Customer Id', 'First Name', 'Subscription Date']].head())

Customers subscribed in 2021 or later: 59
  First Name                     Country Subscription Date
1    Preston                    Djibouti        2021-04-23
4     Joanna  Slovakia (Slovak Republic)        2021-04-17
6     Darren            Pitcairn Islands        2021-08-24
7      Brett                    Bulgaria        2021-04-12
9   Michelle                 Timor-Leste        2021-11-08


Customers from ['Chile', 'Vietnam', 'Bosnia and Herzegovina']:
   First Name                 Country
5       Aimee  Bosnia and Herzegovina
11      Jenna                 Vietnam


Customers from 2020 with email: 36
        Customer Id First Name Subscription Date
2   6F94879bDAfE5a6        Roy        2020-03-25
3   5Cef8BFA16c5e3c      Linda        2020-06-02
5   2d08FB17EE273F4      Aimee        2020-02-25
8   C2dE4dEEc489ae0     Sheryl        2020-01-13
11  CEDec94deE6d69B      Jenna        2020-11-29


## 4. Filtering Rows

In [4]:
# Clean and standardize data
# 1. Remove leading/trailing whitespace from text columns
text_columns = ['First Name', 'Last Name', 'Company', 'City', 'Country']
df_clean[text_columns] = df_clean[text_columns].apply(lambda x: x.str.strip())

# 2. Remove duplicates based on Customer Id
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Customer Id'], keep='first')
print(f"Removed {initial_rows - len(df_clean)} duplicate records")

# 3. Create a subscription year column for analysis
df_clean['Subscription Year'] = df_clean['Subscription Date'].dt.year
df_clean['Subscription Month'] = df_clean['Subscription Date'].dt.month

print("\nDataset after cleaning:")
print(f"Shape: {df_clean.shape}")
print(f"\nFirst 3 rows of cleaned data:")
print(df_clean[['Customer Id', 'First Name', 'Last Name', 'Country', 'Subscription Date']].head(3))

Removed 0 duplicate records

Dataset after cleaning:
Shape: (95, 14)

First 3 rows of cleaned data:
       Customer Id First Name Last Name              Country Subscription Date
1  1Ef7b82A4CAAD10    Preston    Lozano             Djibouti        2021-04-23
2  6F94879bDAfE5a6        Roy     Berry  Antigua and Barbuda        2020-03-25
3  5Cef8BFA16c5e3c      Linda     Olsen   Dominican Republic        2020-06-02


## 3. Data Cleaning - Text & Format Standardization

In [3]:
# Create a copy for cleaning
df_clean = df.copy()

# Introduce some missing values to simulate real-world messy data
np.random.seed(42)
missing_indices = np.random.choice(df_clean.index, size=10, replace=False)
df_clean.loc[missing_indices[:5], 'Phone 2'] = np.nan
df_clean.loc[missing_indices[5:], 'Email'] = np.nan

print("Missing values after introducing NaN:")
print(df_clean.isnull().sum())
print("\n" + "="*60)

# Handle missing values strategies
# 1. Drop rows where Email is missing (critical field)
df_clean = df_clean.dropna(subset=['Email'])

# 2. Fill missing Phone 2 with 'Not provided'
df_clean['Phone 2'] = df_clean['Phone 2'].fillna('Not provided')

# 3. Convert Subscription Date to datetime
df_clean['Subscription Date'] = pd.to_datetime(df_clean['Subscription Date'])

print("After handling missing values:")
print(df_clean.isnull().sum())
print(f"\nRows remaining: {len(df_clean)} (removed {len(df) - len(df_clean)} rows)")

Missing values after introducing NaN:
Index                0
Customer Id          0
First Name           0
Last Name            0
Company              0
City                 0
Country              0
Phone 1              0
Phone 2              5
Email                5
Subscription Date    0
Website              0
dtype: int64

After handling missing values:
Index                0
Customer Id          0
First Name           0
Last Name            0
Company              0
City                 0
Country              0
Phone 1              0
Phone 2              0
Email                0
Subscription Date    0
Website              0
dtype: int64

Rows remaining: 95 (removed 5 rows)


## 2. Introduce and Handle Missing Values

In [2]:
# Load the customers dataset
df = pd.read_csv('customers-100.csv')

print("Dataset Shape:", df.shape)
print("\n" + "="*60)
print("First 5 rows:")
print(df.head())
print("\n" + "="*60)
print("Data Types:")
print(df.dtypes)
print("\n" + "="*60)
print("Missing Values:")
print(df.isnull().sum())

Dataset Shape: (100, 12)

First 5 rows:
   Index      Customer Id First Name Last Name                        Company  \
0      1  DD37Cf93aecA6Dc     Sheryl    Baxter                Rasmussen Group   
1      2  1Ef7b82A4CAAD10    Preston    Lozano                    Vega-Gentry   
2      3  6F94879bDAfE5a6        Roy     Berry                  Murillo-Perry   
3      4  5Cef8BFA16c5e3c      Linda     Olsen  Dominguez, Mcmillan and Do...   
4      5  053d585Ab6b3159     Joanna    Bender       Martin, Lang and Andrade   

                City                     Country                 Phone 1  \
0       East Leonard                       Chile            229.077.5154   
1  East Jimmychester                    Djibouti              5153435776   
2      Isabelborough         Antigua and Barbuda         +1-539-402-0259   
3         Bensonview          Dominican Republic  001-808-617-6467x12895   
4     West Priscilla  Slovakia (Slovak Republic)  001-234-203-0635x76146   

                

## 1. Load and Explore the Dataset

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

# Data Cleaning & Analysis - Customers Dataset

This notebook demonstrates essential data cleaning and manipulation techniques:
- Loading and exploring data
- Handling missing values
- Filtering rows
- GroupBy operations
- Merging datasets
- Creating pivot tables